# Localised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For LLSTM, each partition/local model applies early stopping independently using the same `args_EARLY_STOP_TOL` and `args_MIN_EPOCHS`. The aggregate loss table is for reporting only.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from llstm import (
    Datacollector,
    run_localised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = 'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 0
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'llstm'
args_truncated = None
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

dict_ = Datacollector(pd.read_csv(args_path), lags=args_lags, ts=range(15), fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 21.95it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0     Total             0 2003-03-03 00:00:00  12864.0  12389.0  12155.0   
1     Total             0 2003-03-03 01:00:00  12389.0  12155.0  12072.0   
2     Total             0 2003-03-03 02:00:00  12155.0  12072.0  12162.0   
3     Total             0 2003-03-03 03:00:00  12072.0  12162.0  12569.0   
4     Total             0 2003-03-03 04:00:00  12162.0  12569.0  13238.0   

   lags_45  lags_44  lags_43  lags_42  ...   lags_9   lags_8   lags_7  \
0  12072.0  12162.0  12569.0  13238.0  ...  15177.0  15573.0  16281.0   
1  12162.0  12569.0  13238.0  14191.0  ...  15573.0  16281.0  16842.0   
2  12569.0  13238.0  14191.0  15213.0  ...  16281.0  16842.0  16503.0   
3  13238.0  14191.0  15213.0  15646.0  ...  16842.0  16503.0  15815.0   
4  14191.0  15213.0  15646.0  15653.0  ...  16503.0  15815.0  14745.0   

    lags_6   lags_5   lags_4   lags_3   lags_2   lags_1        y  
0  16842.0  16503.0  15815.0  14745.0

In [3]:
%%time
results = run_localised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
print('Early stop rule: each local model checks its own avg_normalised_loss delta independently.')
local_stop_summary_df = pd.DataFrame(results['local_stop_summary'])
display(local_stop_summary_df.tail())
pd.DataFrame(results['round_logs']).tail()

[Localised] pid=0 epoch 001/100 | avg_normalised_loss=0.009988 | avg_actual_loss=83733.009637 | delta=nan | stopped=False
[Localised] pid=0 epoch 002/100 | avg_normalised_loss=0.005410 | avg_actual_loss=45353.627699 | delta=0.0045781 | stopped=False
[Localised] pid=0 epoch 003/100 | avg_normalised_loss=0.005798 | avg_actual_loss=48608.504123 | delta=0.000388259 | stopped=False
[Localised] pid=0 epoch 004/100 | avg_normalised_loss=0.003995 | avg_actual_loss=33491.646398 | delta=0.00180322 | stopped=False
[Localised] pid=0 epoch 005/100 | avg_normalised_loss=0.004234 | avg_actual_loss=35492.720786 | delta=0.000238699 | stopped=False
[Localised] pid=0 epoch 006/100 | avg_normalised_loss=0.003801 | avg_actual_loss=31862.580934 | delta=0.000433023 | stopped=False
[Localised] pid=0 epoch 007/100 | avg_normalised_loss=0.003343 | avg_actual_loss=28021.201291 | delta=0.000458221 | stopped=False
[Localised] pid=0 epoch 008/100 | avg_normalised_loss=0.003234 | avg_actual_loss=27115.275741 | delta

,partition_id,stopped_early,stop_epoch,stop_reason,early_stop_tol,min_epochs,final_avg_normalised_loss,final_avg_actual_loss,n_train,n_test
5,5,True,52,normalised_loss_delta<1e-05 after min_epochs=20,0.00001,20,0.002140,90.102349,99288,24835
6,6,True,33,normalised_loss_delta<1e-05 after min_epochs=20,0.00001,20,0.003810,50.388053,99288,24835
7,7,True,27,normalised_loss_delta<1e-05 after min_epochs=20,0.00001,20,0.001944,626.844373,99288,24835
8,8,True,20,normalised_loss_delta<1e-05 after min_epochs=20,0.00001,20,0.002092,318.422920,99288,24835
9,9,True,29,normalised_loss_delta<1e-05 after min_epochs=20,0.00001,20,0.001848,287.644983,99288,24835


CPU times: total: 1h 16min 38s
Wall time: 33min 44s


,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,n_active_partitions_reported,n_partitions_stopped_this_epoch
73,74,0.001304,824.386462,0.000015,1,0
74,75,0.001360,859.391523,0.000055,1,0
75,76,0.001324,836.856993,0.000036,1,0
76,77,0.001313,829.547648,0.000012,1,0
77,78,0.001321,834.603497,0.000008,1,1


In [4]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
6,6,24835,7.644921,0.310041,0.306378
7,7,24835,22.296817,0.208964,0.206412
8,8,24835,15.290428,0.198046,0.203114
9,9,24835,14.427243,0.185215,0.189116
10,Overall,248350,23.713321,0.193225,0.195194


,partition_id,n_test,MAE,MASE,RMSSE
6,6,24835,23.281356,0.944178,0.890582
7,7,24835,99.791777,0.935242,0.924299
8,8,24835,70.849026,0.917657,0.888018
9,9,24835,70.187424,0.901056,0.876588
10,Overall,248350,130.514212,0.920111,0.898427


In [5]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = 'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}GEF_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')
def align_S_for_hierarchicalforecast(S, id_col="unique_id"):
    S = S.copy()
    if id_col not in S.columns:
        S = S.reset_index()
        if id_col not in S.columns:
            S = S.rename(columns={S.columns[0]: id_col})
    bottom_ids = [c for c in S.columns if c != id_col]
    agg_ids = [uid for uid in S[id_col].tolist() if uid not in bottom_ids]
    row_order = agg_ids + bottom_ids
    S_aligned = (
        S
        .set_index(id_col)
        .loc[row_order, bottom_ids]
        .reset_index())
    bottom_block = S_aligned[bottom_ids].tail(len(bottom_ids)).to_numpy()
    if not np.allclose(bottom_block, np.eye(len(bottom_ids))):
        raise ValueError("S Error")
    return S_aligned

S = align_S_for_hierarchicalforecast(S)
reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)

recon_results[0].head()

C:\Users\qifei\AppData\Local\Temp\ipykernel_28624\2550965230.py:54: DeprecationWarning: The 'S' parameter is deprecated and will be removed in a future version. Please use 'S_df' instead.
  recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)


,unique_id,ds,y,llstm,llstm/BottomUp,llstm/MinTrace_method-mint_shrink,llstm/MinTrace_method-wls_var,llstm/MinTrace_method-ols,llstm/MinTrace_method-wls_struct
0,Total,2014-06-30 23:00:00,15138.0,15117.722656,15117.323669,15119.653760,15118.746894,15118.185764,15118.799064
1,Total,2014-07-01 00:00:00,13810.0,13792.689453,13767.591492,13758.388823,13766.280564,13786.911585,13773.728684
2,Total,2014-07-01 01:00:00,12979.0,12962.276367,12977.397339,12963.410854,12966.798021,12961.380525,12962.833075
3,Total,2014-07-01 02:00:00,12468.0,12442.132812,12486.323975,12469.374256,12469.299383,12444.876510,12457.071862
4,Total,2014-07-01 03:00:00,12226.0,12199.419922,12222.573120,12210.138764,12211.581012,12200.061017,12205.269666


In [6]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_loc'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='localised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

# LLSTM-specific early stopping diagnostics: each local model has its own stop epoch.
local_stop_summary_path = f'{output_prefix}_local_stop_summary.csv'
round_logs_by_partition_path = f'{output_prefix}_round_logs_by_partition.csv'
pd.DataFrame(results['local_stop_summary']).to_csv(local_stop_summary_path, index=False)
pd.DataFrame(results['round_logs_by_partition']).to_csv(round_logs_by_partition_path, index=False)
output_paths['local_stop_summary'] = local_stop_summary_path
output_paths['round_logs_by_partition'] = round_logs_by_partition_path

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh1_loc_forecasts.csv
  per_series_metrics: fh1_loc_per_series_metrics.csv
  metrics_by_level: fh1_loc_metrics_by_level.csv
  overall_metrics: fh1_loc_overall_metrics.csv
  round_logs: fh1_loc_round_logs.csv
  timing: fh1_loc_timing.csv
  local_stop_summary: fh1_loc_local_stop_summary.csv
  round_logs_by_partition: fh1_loc_round_logs_by_partition.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,86.045452,0.148736,0.156448,1,localised
1,1,top,bu,73.584291,0.127196,0.132001,1,localised
2,1,top,mint_ols,81.761372,0.141330,0.148177,1,localised
3,1,top,mint_shrinkage,71.013818,0.122752,0.127167,1,localised
4,1,top,mint_var,70.940474,0.122626,0.127869,1,localised
5,1,top,naive,527.787525,0.912318,0.895358,1,localised
6,1,top,wls_structure,73.104213,0.126366,0.131993,1,localised
7,2,middle,base,16.512212,0.198549,0.199475,6,localised
8,2,middle,bu,16.757112,0.199490,0.200781,6,localised
9,2,middle,mint_ols,18.403159,0.251051,0.255854,6,localised


In [7]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,23.713321,0.193225,0.195194,10,localised
1,bu,22.614145,0.191636,0.193533,10,localised
2,mint_ols,24.184069,0.221784,0.225750,10,localised
3,mint_shrinkage,21.591462,0.185324,0.187844,10,localised
4,mint_var,21.646730,0.186325,0.189113,10,localised
5,naive,130.514212,0.920111,0.898427,10,localised
6,wls_structure,22.154720,0.193694,0.196322,10,localised


In [8]:
timing_df

,module,seconds
0,training_sec,2023.085728
1,total_sec,2024.082700
